# UNet Training for Kaggle (T4 x 2 Dual GPU)
This notebook performs the following:
1. Splits the provided dataset into `train` and `valid` folders in a 4:1 ratio.
2. Defines a custom PyTorch `Dataset` for Image and Mask loading with **albumentations** augmentation.
3. Implements a standard PyTorch UNet model (raw logits output).
4. Uses `nn.DataParallel` to utilize dual T4 GPUs.
5. Uses `BCEWithLogitsLoss` + `CosineAnnealingLR` scheduler.
6. Computes the Dice Coefficient as the evaluation metric during training and validation.

In [ ]:
!pip install -q albumentations

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

import glob

# Auto-detect DATASET_DIR by searching for metadata.csv recursively
DATASET_DIR = None
search_paths = ['/kaggle/input', '.', '../input', '/content']

for base_path in search_paths:
    if os.path.exists(base_path):
        for root, dirs, files in os.walk(base_path):
            if 'metadata.csv' in files:
                DATASET_DIR = root
                break
    if DATASET_DIR is not None:
        break

if DATASET_DIR is None:
    error_msg = (
        "Could not auto-detect the dataset folder containing 'metadata.csv'.\n"
        "If you are on Kaggle, make sure you have added the dataset to your notebook via 'Add Data'.\n"
        "If you are running locally, please manually set the DATASET_DIR variable below."
    )
    # Uncomment and set manually if needed:
    # DATASET_DIR = '/path/to/your/dataset'
    raise FileNotFoundError(error_msg)

print(f"Detected Dataset Directory: {DATASET_DIR}")

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './working'

IMAGE_DIR = os.path.join(DATASET_DIR, 'Image')
MASK_DIR = os.path.join(DATASET_DIR, 'Mask')
CSV_PATH = os.path.join(DATASET_DIR, 'metadata.csv')

# Load metadata
df = pd.read_csv(CSV_PATH)

# Split 80/20 (more training data for small dataset)
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42)

def create_and_copy(dataframe, subset_name):
    img_out = os.path.join(WORKING_DIR, subset_name, 'Image')
    mask_out = os.path.join(WORKING_DIR, subset_name, 'Mask')
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)
    
    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=f"Copying {subset_name}"):
        img_src = os.path.join(IMAGE_DIR, row['Image'])
        mask_src = os.path.join(MASK_DIR, row['Mask'])
        
        img_dst = os.path.join(img_out, row['Image'])
        mask_dst = os.path.join(mask_out, row['Mask'])
        
        if os.path.exists(img_src) and os.path.exists(mask_src):
            shutil.copy(img_src, img_dst)
            shutil.copy(mask_src, mask_dst)
        else:
            print(f"Warning: Missing file {row['Image']} or {row['Mask']}")

# Create train and valid folders and copy images
print("Splitting data into train and valid folders...")
create_and_copy(train_df, 'train')
create_and_copy(valid_df, 'valid')
print("Data splitting complete.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- Augmentation transforms (albumentations) ---
# ImageNet normalization constants
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-15, 15), p=0.4, border_mode=0),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.CoarseDropout(num_holes_range=(2, 5), hole_height_range=(12, 20), hole_width_range=(12, 20), fill=0, p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

class SegmentationDataset(Dataset):
    def __init__(self, df, subset_name, transform=None):
        self.df = df.reset_index(drop=True)
        self.subset_name = subset_name
        self.transform = transform
        self.img_dir = os.path.join(WORKING_DIR, subset_name, 'Image')
        self.mask_dir = os.path.join(WORKING_DIR, subset_name, 'Mask')
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Image']
        mask_name = self.df.loc[idx, 'Mask']
        
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        image = np.array(Image.open(img_path).convert("RGB").resize((256, 256), Image.BILINEAR))
        mask = np.array(Image.open(mask_path).convert("L").resize((256, 256), Image.NEAREST))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']  # float32 tensor (C, H, W)
            mask = augmented['mask']    # uint8 tensor (H, W)
        
        # Cast mask to float and add channel dim; binarize
        mask = mask.float().unsqueeze(0)  # (1, H, W)
        mask = (mask > 127.0).float()     # binarize (0 or 1)
        
        return image, mask

train_dataset = SegmentationDataset(train_df, 'train', transform=train_transform)
valid_dataset = SegmentationDataset(valid_df, 'valid', transform=valid_transform)

# Using num_workers=4 for faster data loading
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")
print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}")

# --- UNet Model ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, dropout_rate=0.15):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout2d(p=dropout_rate)
        
        self.up1 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv3 = DoubleConv(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, 1)
        
    def forward(self, x):
        x1 = self.down1(x)
        p1 = self.pool(x1)
        
        x2 = self.down2(p1)
        p2 = self.pool(x2)
        
        x3 = self.down3(p2)
        p3 = self.pool(x3)
        
        x4 = self.down4(p3)
        x4 = self.dropout(x4)  # Dropout at bottleneck
        
        u1 = self.up1(x4)
        u1 = torch.cat([u1, x3], dim=1)
        c1 = self.conv1(u1)
        
        u2 = self.up2(c1)
        u2 = torch.cat([u2, x2], dim=1)
        c2 = self.conv2(u2)
        
        u3 = self.up3(c2)
        u3 = torch.cat([u3, x1], dim=1)
        c3 = self.conv3(u3)
        
        # Return RAW LOGITS — no sigmoid here
        return self.out(c3)

# Dice Coefficient (evaluation metric)
def dice_coeff(logits, target, smooth=1e-6):
    """Compute Dice from raw logits. Applies sigmoid internally."""
    pred = (torch.sigmoid(logits) > 0.5).float()
    intersection = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean().item()

# Differentiable Dice Loss (for training)
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2,3))
        union = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3))
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

# Combined BCE + Dice Loss
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        return self.bce_weight * self.bce(logits, targets) + self.dice_weight * self.dice(logits, targets)

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Number of GPUs available: {torch.cuda.device_count()}")

model = UNet(in_channels=3, out_channels=1)

# Use DataParallel if multiple GPUs are available (Perfect for Kaggle T4 x 2)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training!")
    model = nn.DataParallel(model)

model = model.to(device)

# Combined BCE + Dice Loss — directly optimizes the Dice metric
criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 25

# Cosine Annealing: smoothly decays LR from 1e-3 toward 0 over all epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Early stopping: save best model based on validation Dice
best_valid_dice = 0.0
patience = 7
epochs_no_improve = 0

for epoch in range(num_epochs):
    model.train()
    train_dice = 0
    
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        train_dice += dice_coeff(outputs, masks)
        
    model.eval()
    valid_dice = 0
    
    with torch.no_grad():
        for images, masks in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]", leave=False):
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            valid_dice += dice_coeff(outputs, masks)
    
    # Step the scheduler once per epoch
    scheduler.step()
            
    train_dice /= len(train_loader)
    valid_dice /= len(valid_loader)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.6f}")
    print(f"Train Dice: {train_dice:.4f} | Valid Dice: {valid_dice:.4f}")
    
    # Early stopping check
    if valid_dice > best_valid_dice:
        best_valid_dice = valid_dice
        torch.save(model.state_dict(), 'unet_best.pth')
        print(f">> New best model saved! (Valid Dice: {best_valid_dice:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {patience} epochs with no improvement.")
            break
    
    print("-" * 50)

# Load best model
model.load_state_dict(torch.load('unet_best.pth'))
print(f"\nTraining complete. Best Valid Dice: {best_valid_dice:.4f}")
print("Best model loaded from unet_best.pth")

## Visualizing Model Performance
Let's see how the model actually performs by plotting original images, ground truth masks, and the predicted masks side-by-side.

In [ ]:
import matplotlib.pyplot as plt

def denormalize(tensor, mean=MEAN, std=STD):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

def visualize_predictions(model, dataloader, device, num_samples=5):
    model.eval()
    samples_shown = 0
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            preds = (torch.sigmoid(logits) > 0.5).float()
            
            for i in range(images.size(0)):
                if samples_shown >= num_samples:
                    break
                
                # Denormalize image for correct colors
                img = denormalize(images[i]).permute(1, 2, 0).numpy()
                mask = masks[i].cpu().squeeze().numpy()
                pred = preds[i].cpu().squeeze().numpy()
                
                axes[samples_shown, 0].imshow(img)
                axes[samples_shown, 0].set_title("Original Image")
                axes[samples_shown, 0].axis('off')
                
                axes[samples_shown, 1].imshow(mask, cmap='gray')
                axes[samples_shown, 1].set_title("Ground Truth Mask")
                axes[samples_shown, 1].axis('off')
                
                axes[samples_shown, 2].imshow(pred, cmap='gray')
                axes[samples_shown, 2].set_title("Predicted Mask")
                axes[samples_shown, 2].axis('off')
                
                samples_shown += 1
            
            if samples_shown >= num_samples:
                break
    
    plt.tight_layout()
    plt.show()

# Explicitly reload best model before visualizing
print("Loading best model from unet_best.pth...")
model.load_state_dict(torch.load('unet_best.pth', map_location=device))
model.eval()
print("Best model loaded.\n")

# Visualize 5 samples from the validation set
visualize_predictions(model, valid_loader, device, num_samples=5)

In [ ]:
# ── Load the best model explicitly ──
best_model = UNet(in_channels=3, out_channels=1)

# Handle DataParallel saved weights
state_dict = torch.load('unet_best.pth', map_location=device)
# If saved with DataParallel, keys start with 'module.' — strip that prefix
from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace('module.', '') if k.startswith('module.') else k
    new_state_dict[name] = v
best_model.load_state_dict(new_state_dict)

# Wrap in DataParallel if multiple GPUs
if torch.cuda.device_count() > 1:
    best_model = nn.DataParallel(best_model)
best_model = best_model.to(device)
best_model.eval()
print(f'Best model loaded from unet_best.pth on {device}')

# ── Inference on validation set with flood overlay ──
def test_best_model(model, dataloader, device, num_samples=6):
    model.eval()
    samples_shown = 0
    all_dice = []
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(18, 6 * num_samples))
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            
            # Compute dice for this batch
            batch_dice = dice_coeff(logits, masks)
            all_dice.append(batch_dice)
            
            for i in range(images.size(0)):
                if samples_shown >= num_samples:
                    break
                
                # Denormalize image
                img = denormalize(images[i]).permute(1, 2, 0).numpy()
                mask_gt = masks[i].cpu().squeeze().numpy()
                pred_mask = preds[i].cpu().squeeze().numpy()
                
                # Create colored overlay (red = predicted flood)
                overlay = img.copy()
                overlay[pred_mask == 1] = overlay[pred_mask == 1] * 0.4 + np.array([1.0, 0.0, 0.0]) * 0.6
                
                # Per-sample dice
                inter = (pred_mask * mask_gt).sum()
                union = pred_mask.sum() + mask_gt.sum()
                sample_dice = (2 * inter + 1e-6) / (union + 1e-6)
                
                axes[samples_shown, 0].imshow(img)
                axes[samples_shown, 0].set_title('Original Image', fontsize=14)
                axes[samples_shown, 0].axis('off')
                
                axes[samples_shown, 1].imshow(mask_gt, cmap='gray')
                axes[samples_shown, 1].set_title('Ground Truth Mask', fontsize=14)
                axes[samples_shown, 1].axis('off')
                
                axes[samples_shown, 2].imshow(overlay)
                axes[samples_shown, 2].set_title(f'Prediction Overlay (Dice: {sample_dice:.4f})', fontsize=14)
                axes[samples_shown, 2].axis('off')
                
                samples_shown += 1
            
            if samples_shown >= num_samples:
                break
    
    plt.tight_layout()
    plt.show()
    
    avg_dice = sum(all_dice) / len(all_dice)
    print(f'\nOverall Validation Dice (best model): {avg_dice:.4f}')

test_best_model(best_model, valid_loader, device, num_samples=6)

In [ ]:
import shutil

# Tool to zip the folder
# Syntax: shutil.make_archive('output_zip_name', 'zip', 'path_to_output_folder')
shutil.make_archive('kaggle_output', 'zip', '/kaggle/working/')

print("Folder zipped successfully!")